# Benchmarking transformation methods

## Import packages

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xwmt
import xbudget
import xhistogram
from xhistogram.xarray import histogram
import os

# Optional (loading to memory and plotting) 
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import calendar

In [ ]:
print(
    'numpy version',np.__version__, '\npandas version',pd.__version__,
    '\nxarray version',xr.__version__, '\nxhistogram version',xhistogram.__version__, '\nxgcm version',xgcm.__version__,
    '\nxwmt version',xwmt.__version__, '\nxbudget version',xbudget.__version__,)

### Loading a dataset

This notebook loads GFDL-CM4 output directly from the public [Pangeo CMIP6 archive on Google Cloud Storage](https://console.cloud.google.com/storage/browser/cmip6) (anonymous, read-only access via `gcsfs`), so it can be run by anyone without access to GFDL's internal filesystem. This is the **same published ESGF/CMIP6 data** that GFDL also serves locally, so for identical facets and time period the values are bit-for-bit identical.

All relevant variables are loaded into `ds` and the (static) grid fields are loaded into a separate list. The Pangeo zarr stores span the full 1850–2014 historical record; we subset to the last five years (Jan 2010 – Dec 2014) of the CM4 historical run via `time_slice`, matching the original `201001-201412` files. Widen `time_slice` (e.g. `slice(None)`) to use more years.

Notes on data availability for these facets on Pangeo:
- `sfdsi` (surface salt flux) is not published to the Pangeo archive, so it is skipped (it was also absent from the original GFDL-local files).
- The static `basin` and `deptho` fields are only published under the `piControl` experiment on Pangeo. Since they are time-invariant model geometry, they are identical to the historical static fields.

In [ ]:
import gcsfs  # registers the gs:// filesystem so xarray can open the zarr stores

# Public Pangeo CMIP6 archive on Google Cloud Storage.
gcs_root = 'gs://cmip6/CMIP6'
activity_id = 'CMIP'
institution_id = 'NOAA-GFDL'
source_id = 'GFDL-CM4'
experiment_id = 'historical'
member_id = 'r1i1p1f1'
table_id = 'Omon'
grid_label = 'gn'
version = 'v20180701'
storage_options = {'token': 'anon'} # anonymous, read-only access

# Subset to the last five years (Jan 2010 - Dec 2014) to match the original
# `201001-201412` files. Set `time_slice = slice(None)` to use the full record.
time_slice = slice('2010-01-01', '2014-12-31')

def cmip6_zstore(variable_id, table_id=table_id, experiment_id=experiment_id):
    return '/'.join([
        gcs_root, activity_id, institution_id, source_id, experiment_id,
        member_id, table_id, variable_id, grid_label, version
    ]) + '/'

In [ ]:
variables = ['tos','sos','hfds','wfo','sfdsi']
chunks = {'time':1, 'x':-1, 'y':-1, 'xh':-1, 'yh':-1}

ds = xr.Dataset()
for var in variables:
    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        da = xr.open_dataset(
            cmip6_zstore(var), engine='zarr', decode_times=time_coder, chunks=chunks,
            backend_kwargs={'storage_options': storage_options}
        )[var]
        print('Loading', var)
        ds[var] = da.sel(time=time_slice)
    except Exception:
        print('Store for', var, 'is not available on Pangeo. Skipping.')

In [ ]:
# Static grid fields. `areacello` is published for the historical experiment, but
# `deptho` and `basin` are only published once (under piControl) on Pangeo. These are
# time-invariant model-geometry fields and are identical across experiments.
grid = []
static_experiment = {'areacello': experiment_id, 'deptho': 'piControl', 'basin': 'piControl'}
for var, expt in static_experiment.items():
    try:
        print('Loading', var, '(' + expt + ')')
        grid.append(xr.open_dataset(
            cmip6_zstore(var, table_id='Ofx', experiment_id=expt), engine='zarr',
            chunks=chunks, backend_kwargs={'storage_options': storage_options}
        ))
    except Exception:
        print('Store for', var, 'is not available on Pangeo. Skipping.')

In [ ]:
ds = xr.merge([ds, xr.merge(grid[1:])])

# Area needs to be loaded seperately after renaming MOM6-specific dimension names (xh, yh) 
ds['areacello'] = grid[0].areacello.rename({'xh': 'x', 'yh': 'y'})

In [ ]:
# Add core coordinates of ocean_grid to ds
ds = ds.assign_coords({
    "areacello": xr.DataArray(ds["areacello"].values, dims=('y', 'x',)), # Required for area-integration
    "lon":       xr.DataArray(ds["lon"].values, dims=('y', 'x',)), # Required for calculating density if not already provided!
    "lat":       xr.DataArray(ds["lat"].values, dims=('y', 'x',)), # Required for calculating density if not already provided!
})

# xgcm grid for dataset
coords = {
    'X': {'center': 'x',},
    'Y': {'center': 'y',},
}
metrics = {
    ('X','Y'): "areacello", # Required for area-integration
}
grid = xgcm.Grid(ds, coords=coords, metrics=metrics, boundary={'X':'periodic', 'Y':'periodic'}, autoparse_metadata=False)

In [ ]:
grid._ds['hfds'].mean('time').plot(robust=True)

In [ ]:
budgets_dict = {
    "mass": {},
    "heat": {"surface_lambda": "tos"},
    "salt": {"surface_lambda": "sos"}
}

In [ ]:
swmt = xwmt.WaterMassTransformations(grid, budgets_dict)

In [ ]:
bins = xr.DataArray(np.arange(-4., 50., 1.), dims=('tos_bin'))

### Benchmarking `integrate_transformations`

In [ ]:
%%time
tmp_xhistogram_global = histogram(
    swmt.grid._ds['tos'].expand_dims('z_l'),
    bins=[bins.values],
    dim=['x', 'y', 'z_l'],
    weights=swmt.grid._ds['hfds'].fillna(0.).expand_dims('z_l'),
).mean('time')
tmp_xhistogram_global.load();
;

In [ ]:
%%time
tmp_xgcm = swmt.grid.transform(
    swmt.grid._ds['hfds'].fillna(0.).expand_dims('z_l'),
    "Z",
    target=bins,
    target_data=swmt.grid._ds['tos'].expand_dims({'z_i': xr.DataArray([0,1], dims=('z_i',))}),
    method="conservative",
    
).fillna(0.).sum(['x','y']).mean('time')
tmp_xgcm.load();
;

In [ ]:
%%time
tmp_xhistogram_sumlocal = histogram(
    swmt.grid._ds['tos'].expand_dims('z_l'),
    bins=[bins.values],
    dim=['z_l'],
    weights=swmt.grid._ds['hfds'].fillna(0.).expand_dims('z_l'),
).sum(['x', 'y']).mean('time')
tmp_xhistogram_sumlocal.load();
;

In [ ]:
plt.figure(figsize=(10, 3))
plt.subplot(1,2,1)
tmp_xhistogram_global.plot(label="xhistogram", color="k")
tmp_xgcm.plot(label="xgcm", linestyle="--", color="r")
tmp_xhistogram_sumlocal.plot(label="xhistogram", linestyle=":", color="b")
plt.title("Global versus local-then-summed `xhistogram` calculations")

plt.subplot(1,2,2)
((tmp_xhistogram_global - tmp_xhistogram_sumlocal.values)/tmp_xhistogram_global).plot(label="xhistogram sum local")
((tmp_xhistogram_global - tmp_xgcm.values)/tmp_xhistogram_global).plot(label="xgcm")
plt.title("Error relative to global `xhistogram` calculation")
plt.ylabel("relative error")
plt.legend()
plt.tight_layout()

print()
print(
    "Global sum to assess heat conservation:\n"
    "- Raw surface flux:", grid._ds['hfds'].mean('time').sum().values,"\n",
    "- Global xhistogram:", tmp_xhistogram_global.sum().values,"\n",
    "- Local xgcm", tmp_xgcm.sum().values,"\n",
    "- Local xhistogram", tmp_xhistogram_sumlocal.sum().values
)

### Benchmarking `map_transformations` for a single slice

In [ ]:
tos_lev = 12.5

In [ ]:
%%time
tmp_xgcm_isosurface = swmt.grid.transform(
    swmt.grid._ds['hfds'].fillna(0.).expand_dims({'z_l': xr.DataArray([0.5], dims=('z_l',))}),
    "Z",
    target=bins,
    target_data=swmt.grid._ds['tos'].expand_dims({'z_i': xr.DataArray([0,1], dims=('z_i',))}),
    method="conservative"
).sel({"tos_bin":tos_lev}, method="nearest").fillna(0.).mean('time')
tmp_xgcm_isosurface.load();

In [ ]:
%%time
tmp_xhistogram_isosurface = histogram(
    swmt.grid._ds['tos'].expand_dims({'z_l': xr.DataArray([0.5], dims=('z_l',))}),
    bins=[bins.values],
    dim=['z_l'],
    weights=swmt.grid._ds['hfds'].fillna(0.).expand_dims({'z_l': xr.DataArray([0.5], dims=('z_l',))}),
).sel({"tos_bin":tos_lev}, method="nearest").mean('time')
tmp_xhistogram_isosurface.load();

In [ ]:
plt.figure(figsize=(16,4))
plt.subplot(1,3,1)
tmp_xgcm_isosurface.plot(vmin=-10, vmax=10, cmap="RdBu_r")
plt.subplot(1,3,2)
tmp_xhistogram_isosurface.plot(vmin=-10, vmax=10, cmap="RdBu_r")
plt.subplot(1,3,3)
(tmp_xgcm_isosurface - tmp_xhistogram_isosurface.values).plot(vmin=-1.e-5, vmax=1.e-5, cmap="RdBu");

### Benchmarking `map_transformations` for full 3D fields

In [ ]:
%%time
tmp_xgcm_isosurfaces = swmt.grid.transform(
    swmt.grid._ds['hfds'].fillna(0.).expand_dims('z_l'),
    "Z",
    target=bins,
    target_data=swmt.grid._ds['tos'].expand_dims({'z_i': xr.DataArray([0,1], dims=('z_i',))}),
    method="conservative"
).fillna(0.).mean('time')
tmp_xgcm_isosurfaces.load();
;

In [ ]:
%%time
tmp_xhistogram_isosurfaces = histogram(
    swmt.grid._ds['tos'].expand_dims('z_l'),
    bins=[bins.values],
    dim=['z_l'],
    weights=swmt.grid._ds['hfds'].fillna(0.).expand_dims('z_l'),
).mean('time')
tmp_xhistogram_isosurfaces.load();
;

There are some very minor discrepancies between the two regridding methods. They are not noticeable by eye, but the coordinate transformations differ quantitatively in a handful of cells. I do not yet understand why that would be.

In [ ]:
Tmax = (tmp_xhistogram_isosurfaces!=tmp_xgcm_isosurfaces).sum(['x','y']).idxmax()

In [ ]:
(tmp_xhistogram_isosurfaces!=tmp_xgcm_isosurfaces).sum(['x','y'])

In [ ]:
plt.figure(figsize=(16,4))
plt.subplot(1,3,1)
tmp_xgcm_isosurfaces.sel(tos_bin=Tmax).plot(robust=True)
plt.title("`xgcm`")
plt.subplot(1,3,2)
tmp_xhistogram_isosurfaces.sel(tos_bin=Tmax).plot(robust=True)
plt.title("`xhistogram`")
plt.subplot(1,3,3)
(tmp_xgcm_isosurfaces.sel(tos_bin=Tmax) - tmp_xhistogram_isosurfaces.sel(tos_bin=Tmax).values).plot(vmin=-1.e-5, vmax=1.e-5, cmap="RdBu")
plt.ylim(-10, 15)
plt.xlim(-290, -280)
plt.title("Zooming in on difference pixels");

### Takeaways from benchmarking analysis

Assuming these results scale similarly for larger calculations, these results suggest that `xhistogram` should be the default method for the area-integrated `integrate_transformations` method of `xwmt.WaterMassTransformations`, while `xgcm` should be the default method for `map_transformations`.